In [ ]:
%run ../shared_notebook_setup.py

In [ ]:
messages=[
    {
      "role": "user",
      "content": "Hello World!",
    }
  ]

chat_completion = client.chat.completions.create(
  messages=messages,
  model=DATABRICKS_ENDPOINT
)

In [3]:

print(chat_completion.choices[0].message.content)

Hello World!


### Zero Shot Prompting

In [4]:
# Zero-shot prompting: no examples provided, model relies on pre-trained knowledge
zero_shot_messages = [
    {
        "role": "user",
        "content": "Classify the sentiment of this review as Positive, Negative, or Neutral:\n'The battery life is terrible and it broke after a week.'"
    }
]

zero_shot_response = client.chat.completions.create(
    messages=zero_shot_messages,
    model=DATABRICKS_ENDPOINT
)
print("=== Zero-Shot Prompting ===")
print(zero_shot_response.choices[0].message.content)

=== Zero-Shot Prompting ===
The sentiment of this review is Negative.

The reviewer explicitly states that the battery life is "terrible" and that the product "broke after a week", indicating a strong negative opinion. There is no positive or neutral commentary in the review. Therefore, the sentiment is Negative.


### Few Shot Prompting

In [10]:
# Few-shot prompting: a few examples are provided to guide the model
few_shot_messages = [
    {
        "role": "user",
        "content": """Classify the sentiment of the review as Positive, Negative, or Neutral.

Review: 'This product exceeded all my expectations!'
Sentiment: Positive

Review: 'It was great.'
Sentiment: Positive

Review: 'The battery life is ok'
Sentiment:"""
    }
]

few_shot_response = client.chat.completions.create(
    messages=few_shot_messages,
    model=DATABRICKS_ENDPOINT
)
print("\n=== Few-Shot Prompting ===")
print(few_shot_response.choices[0].message.content)


=== Few-Shot Prompting ===
Neutral.

The sentiment is Neutral because the reviewer states that the battery life is "ok", which implies a mediocre or average experience, rather than expressing a strongly positive or negative opinion.


### Prompt Chaining

In [ ]:
# Prompt chaining example: Step 1 (extract key issues), Step 2 (final sentiment from extracted issues)

review_text = "The phone has a great screen and fast performance, but the battery drains quickly and the camera is mediocre."

# Step 1: Ask the model to extract structured signals
chain_step1_messages = [
    {
        "role": "user",
        "content": f"""
Extract key signals from this product review.

Return exactly in this format:
Positives: <comma-separated list>
Negatives: <comma-separated list>

Review: "{review_text}"
"""
    }
]

chain_step1_response = client.chat.completions.create(
    messages=chain_step1_messages,
    model=DATABRICKS_ENDPOINT
)

extracted_signals = chain_step1_response.choices[0].message.content
print("=== Chain Step 1: Extracted Signals ===")
print(extracted_signals)


In [ ]:

# Step 2: Use the output from Step 1 as input to a second prompt
chain_step2_messages = [
    {
        "role": "user",
        "content": f"""
Using only the extracted signals below, classify overall sentiment as Positive, Negative, or Neutral.
Then provide one short explanation.

Extracted signals:
{extracted_signals}
"""
    }
]

chain_step2_response = client.chat.completions.create(
    messages=chain_step2_messages,
    model=DATABRICKS_ENDPOINT
)

print("\n=== Chain Step 2: Final Sentiment ===")
print(chain_step2_response.choices[0].message.content)

### Chain of thought 

In [ ]:
# Chain-of-thought prompting example
cot_messages = [
    {
        "role": "user",
        "content": f"""
Analyze the sentiment of the review below.

Think step by step before giving the final label.
Return in this exact format:

Reasoning:
1) ...
2) ...
3) ...

Final Sentiment: <Positive, Negative, or Neutral>

Review: "{review_text}"
"""
    }
]

cot_response = client.chat.completions.create(
    messages=cot_messages,
    model=DATABRICKS_ENDPOINT
)

print("=== Chain-of-Thought Prompting ===")
print(cot_response.choices[0].message.content)